# EQ2425 Project 2: Visual search system

In [ ]:
import cv2 as cv
import numpy as np
import os
import matplotlib.pyplot as plt
from collections import Counter

## Section 2: Feature extraction

In [ ]:
folder_path = os.path.join("Data2", "server")

folder_path_client = os.path.join("Data2", "client")

def extracting_features(folderpath):

    image_list = []
    object_numbers = []          # object number of every image, read from the file name

    for filename in sorted(os.listdir(folderpath)):
        path = os.path.join(folderpath, filename)

        image = cv.imread(path, cv.IMREAD_GRAYSCALE)

        if image is not None:
            image_list.append(image)
            object_numbers.append(int(filename.split("_")[0][3:]))   # "obj13_2.JPG" -> 13

    print("Number of images in the folder ", len(image_list))

    Sift = cv.SIFT_create(nfeatures = 3000)

    descriptors_list = []
    for image in image_list:

        keypoints, descriptors = Sift.detectAndCompute(image, None)

        descriptors_list.append(descriptors)

    return descriptors_list, object_numbers


def dictionary(descriptors_list, object_numbers):
    descriptor_dictionary = {}

    for descriptors, obj in zip(descriptors_list, object_numbers):

        if obj in descriptor_dictionary:
            descriptor_dictionary[obj] = np.concatenate((descriptor_dictionary[obj], descriptors))
        else:
            descriptor_dictionary[obj] = descriptors

    return descriptor_dictionary

In [ ]:
server_image_descriptors, server_object_numbers = extracting_features(folder_path)

server_image_dictionary = dictionary(server_image_descriptors, server_object_numbers)

client_image_descriptors, client_object_numbers = extracting_features(folder_path_client)

client_image_dictionary = dictionary(client_image_descriptors, client_object_numbers)

In [ ]:
print("Objects in server / client:", len(server_image_dictionary), len(client_image_dictionary))
print("obj26 (2 images):", len(server_image_dictionary[26]), "  obj37 (4 images):", len(server_image_dictionary[37]))

print("2(a) average features per database object:", np.mean([len(d) for d in server_image_dictionary.values()]))
print("2(b) average features per query object:   ", np.mean([len(d) for d in client_image_dictionary.values()]))

## Section 3: Vocabulary tree

In [ ]:
# ============================================================
# STEP A - Prepare descriptors for vocabulary tree
# ============================================================

all_descriptors = []
object_ids = []

for object_id, descriptors in server_image_dictionary.items():

    all_descriptors.append(descriptors)
    object_ids.extend([object_id] * len(descriptors))


all_descriptors = np.vstack(all_descriptors).astype(np.float32)
object_ids = np.array(object_ids, dtype=np.int32)


print("Shape of all descriptors:", all_descriptors.shape)
print("Shape of object IDs:", object_ids.shape)
print("Number of descriptors:", len(all_descriptors))
print("Number of object IDs:", len(object_ids))


print("Same number of descriptors and object IDs:",
      len(all_descriptors) == len(object_ids))

In [ ]:
# ============================================================
# STEP B - Hierarchical K-Means Vocabulary Tree
# ============================================================

def hi_kmeans(data, b, depth):

    indices = np.arange(len(data))
    leaves = []

    tree = build_tree(data, indices, b, depth, 0, leaves)

    tree["leaves"] = leaves

    return tree


def build_tree(data, indices, b, depth, current_depth, leaves):
    if current_depth == depth or len(data) < b:

        leaf = {
            "is_leaf": True,
            "depth": current_depth,
            "indices": indices,
            "leaf_id": len(leaves)          # number of this visual word
        }
        leaves.append(leaf)
        return leaf

    data_float = data.astype(np.float32)

    criteria = (
        cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER,
        20,                                  # max iterations
        1.0
    )
    compactness, labels, centers = cv.kmeans(
        data_float,
        b,
        None,
        criteria,
        10,                                  # attempts
        cv.KMEANS_PP_CENTERS
    )

    labels = labels.ravel()
    node = {
        "is_leaf": False,
        "depth": current_depth,
        "centroids": centers,               # 3(a): the b centers stored in the node
        "children": []
    }

    for cluster_id in range(b):

        mask = (labels == cluster_id)
        child_data = data[mask]
        child_indices = indices[mask]

        child = build_tree(
            child_data,
            child_indices,
            b,
            depth,
            current_depth + 1,
            leaves
        )

        node["children"].append(child)

    return node

In [ ]:
# ============================================================
# STEP B2 - Number the leaves (visual words) and check the tree
# ============================================================

def get_leaves(node, leaves):

    if node["is_leaf"]:
        node["leaf_id"] = len(leaves)
        leaves.append(node)
    else:
        for child in node["children"]:
            get_leaves(child, leaves)

    return leaves

tree = hi_kmeans(all_descriptors, 4, 3)
leaves = get_leaves(tree, [])

print("Number of leaves:", len(leaves), " (4^3 =", 4**3, ")")
print("Descriptors in all leaves:", sum(len(leaf["indices"]) for leaf in leaves))
print("Descriptors in total:     ", len(all_descriptors))

### Look at the tree

Each dot is a node. The bigger the dot, the more descriptors it holds. The root at the top holds all of them, k-means splits them into b groups per level, and the bottom row are the leaves (visual words).

In [ ]:
def draw_tree(node, ax, counter, level=0):

    if node["is_leaf"]:
        x = counter[0]                     # next free position on the bottom row
        counter[0] += 1
        n = len(node["indices"])
    else:
        child_x = []
        n = 0
        for child in node["children"]:
            cx, cn = draw_tree(child, ax, counter, level + 1)
            child_x.append(cx)
            n += cn
        x = np.mean(child_x)
        for cx in child_x:
            ax.plot([x, cx], [-level, -(level + 1)], color="gray", linewidth=0.8, zorder=1)

    ax.scatter(x, -level, s=20 + 400 * n / len(all_descriptors) * (level + 1), color="C0", zorder=2)
    if level <= 1:
        ax.text(x, -level + 0.15, f"{n:,}", ha="center", fontsize=9)
    return x, n


fig, ax = plt.subplots(figsize=(14, 5))
draw_tree(tree, ax, [0])
ax.set_yticks([0, -1, -2, -3], ["root", "level 1", "level 2", "level 3 (leaves)"])
ax.set_xticks([])
ax.set_title("Vocabulary tree b=4, depth=3: numbers = descriptors in the node")
plt.show()

In [ ]:
leaf_sizes = [len(leaf["indices"]) for leaf in leaves]

plt.figure(figsize=(14, 3.5))
plt.bar(range(len(leaf_sizes)), leaf_sizes)
plt.xlabel("leaf number (visual word)")
plt.ylabel("number of descriptors")
plt.title("Descriptors per leaf, b=4, depth=3")
plt.show()

In [ ]:
trees = {}
for b, depth in [(4, 3), (4, 5), (5, 7)]:
    t = hi_kmeans(all_descriptors, b, depth)
    l = get_leaves(t, [])
    trees[(b, depth)] = (t, l)
    print(f"b={b} depth={depth}: {len(l):>6} leaves (max {b**depth:>6}),",
          "all descriptors in leaves:", sum(len(leaf["indices"]) for leaf in l) == len(all_descriptors))

In [ ]:
def add_tfidf_info(tree, object_ids, num_objects):

    for leaf in tree["leaves"]:

        leaf_object_ids = object_ids[leaf["indices"]]

        unique_objects, counts = np.unique(
            leaf_object_ids,
            return_counts=True
        )

        tf = {}

        for object_id, count in zip(unique_objects, counts):
            tf[int(object_id)] = int(count)


        df = len(unique_objects)
        idf = np.log(1+ num_objects / df)

        leaf["tf"] = tf
        leaf["df"] = df
        leaf["idf"] = idf

In [ ]:
for (b, depth), (t, l) in trees.items():
    add_tfidf_info(t, object_ids, len(server_image_dictionary))

add_tfidf_info(tree, object_ids, len(server_image_dictionary))
for leaf in tree["leaves"]:
    print(leaf["leaf_id"], leaf["idf"])

## Section 4: Querying

Each query image is sent down the tree and compared with all 50 database objects using TF-IDF and cosine similarity. Top-1 / top-5 recall: share of queries whose own object is ranked first / in the first five.

In [ ]:
def find_leaf(descriptor, node):
    if node["is_leaf"]:
        return node
    distances = np.sum((node["centroids"] - descriptor)**2, axis=1)
    return find_leaf(descriptor, node["children"][np.argmin(distances)])

def calculate_scores(client_descriptors, tree, num_objects=50):
    leaf_ids = [find_leaf(desc, tree)["leaf_id"] for desc in client_descriptors]
    query_tf_raw = Counter(leaf_ids)
    
    scores = np.zeros(num_objects)
    query_norm_sq = 0.0
    db_norm_sq = np.zeros(num_objects)

    for leaf in tree["leaves"]:
        idf = leaf["idf"]
        for object_id, tf_db in leaf["tf"].items():
            db_norm_sq[object_id - 1] += (tf_db * idf) ** 2
            
    db_norms = np.sqrt(db_norm_sq)
    db_norms[db_norms == 0] = 1.0 
    
    for leaf_id, tf_query in query_tf_raw.items():
        leaf = tree["leaves"][leaf_id]
        idf = leaf["idf"]
        
        query_weight = tf_query * idf
        query_norm_sq += query_weight ** 2
        
        for object_id, tf_database in leaf["tf"].items():
            database_weight = tf_database * idf
            scores[object_id - 1] += query_weight * database_weight
            
    query_norm = np.sqrt(query_norm_sq) if query_norm_sq > 0 else 1.0
    scores = scores / (query_norm * db_norms)
            
    return scores

def evaluate_recall(client_descriptors_list, client_object_numbers, tree, num_objects=50):
    top1_correct = 0
    top5_correct = 0
    total_queries = len(client_descriptors_list)

    for i in range(total_queries):
        descriptors = client_descriptors_list[i]
        expected_id = client_object_numbers[i]
        
        scores = calculate_scores(descriptors, tree, num_objects)
        
        ranked_objects = np.argsort(scores)[::-1] + 1
        
        top5_predictions = ranked_objects[:5]
        top1_prediction = top5_predictions[0]
        
        if top1_prediction == expected_id:
            top1_correct += 1

        if expected_id in top5_predictions:
            top5_correct += 1

    recall_top1 = top1_correct / total_queries
    recall_top5 = top5_correct / total_queries
    
    return recall_top1, recall_top5

In [ ]:
# 4(a): test on the tree b=4, depth=3
results_4a = {}
recall_1, recall_5 = evaluate_recall(client_image_descriptors, client_object_numbers, trees[(4,3)][0])
results_4a[(4, 3)] = (recall_1, recall_5)   # kept for the figures

print("Results for b=4, depth=3:")
print(f"Top-1 Recall: {recall_1 * 100:.2f}% ({int(recall_1 * 50)} of 50)")
print(f"Top-5 Recall: {recall_5 * 100:.2f}% ({int(recall_5 * 50)} of 50)")

In [ ]:
# 4(a): test on the tree b=4, depth=5
recall_1, recall_5 = evaluate_recall(client_image_descriptors, client_object_numbers, trees[(4,5)][0])
results_4a[(4, 5)] = (recall_1, recall_5)   # kept for the figures

print("Results for b=4, depth=5:")
print(f"Top-1 Recall: {recall_1 * 100:.2f}% ({int(recall_1 * 50)} of 50)")
print(f"Top-5 Recall: {recall_5 * 100:.2f}% ({int(recall_5 * 50)} of 50)")

In [ ]:
# 4(a): test on the tree b=5, depth=7
recall_1, recall_5 = evaluate_recall(client_image_descriptors, client_object_numbers, trees[(5,7)][0])
results_4a[(5, 7)] = (recall_1, recall_5)   # kept for the figures

print("Results for b=5, depth=7:")
print(f"Top-1 Recall: {recall_1 * 100:.2f}% ({int(recall_1 * 50)} of 50)")
print(f"Top-5 Recall: {recall_5 * 100:.2f}% ({int(recall_5 * 50)} of 50)")

In [ ]:
def get_random_fraction_descriptors(descriptors_list, fraction, seed=42):
    np.random.seed(seed)  # fixed seed so the results can be reproduced
    reduced_list = []
    
    for desc in descriptors_list:
        n_total = len(desc)
        n_sample = max(1, int(n_total * fraction))
        
        # pick n_sample random indices, without repeats
        random_indices = np.random.choice(n_total, size=n_sample, replace=False)
        reduced_list.append(desc[random_indices])
        
    return reduced_list


tree_b5_d7 = trees[(5, 7)][0]
results_4b = {}   # kept for the figures

for p in [0.90, 0.70, 0.50]:
    
    client_sub = get_random_fraction_descriptors(client_image_descriptors, p, seed=42)
    
    r1, r5 = evaluate_recall(client_sub, client_object_numbers, tree_b5_d7)
    
    results_4b[p] = (r1, r5)
    print(f"Random features used: {int(p * 100)}% -> Top-1 Recall: {r1 * 100:.2f}% | Top-5 Recall: {r5 * 100:.2f}%")

In [ ]:
os.makedirs("figures", exist_ok=True)
tree_list = [(4, 3), (4, 5), (5, 7)]
x = np.arange(3)

image = cv.imread(os.path.join(folder_path, "obj1_1.JPG"), cv.IMREAD_GRAYSCALE)
keypoints = cv.SIFT_create(nfeatures=3000).detect(image, None)
drawn = cv.cvtColor(image, cv.COLOR_GRAY2RGB)
for kp in sorted(keypoints, key=lambda k: -k.response)[:300]:
    cv.circle(drawn, (int(kp.pt[0]), int(kp.pt[1])), int(kp.size * 2) + 10, (0, 255, 0), 5)
plt.figure(figsize=(8, 6))
plt.imshow(drawn)
plt.axis("off")
plt.savefig("figures/sift_keypoints.png", dpi=200, bbox_inches="tight")
plt.show()

objects = sorted(server_image_dictionary)
server_counts = [len(server_image_dictionary[o]) for o in objects]
client_counts = [len(client_image_dictionary[o]) for o in objects]
plt.figure(figsize=(12, 3.5))
plt.bar(np.array(objects) - 0.2, server_counts, 0.4, label="database")
plt.bar(np.array(objects) + 0.2, client_counts, 0.4, label="query")
plt.xlabel("object number")
plt.ylabel("number of SIFT features")
plt.xlim(0.3, 50.7)
plt.legend()
plt.savefig("figures/features_per_object.png", dpi=200, bbox_inches="tight")
plt.show()


fig, ax = plt.subplots(figsize=(14, 5))
draw_tree(trees[(4, 3)][0], ax, [0])
ax.set_yticks([0, -1, -2, -3], ["root", "level 1", "level 2", "level 3 (leaves)"])
ax.set_xticks([])
ax.set_ylim(-3.4, 0.6)
plt.savefig("figures/tree_b4_d3.png", dpi=200, bbox_inches="tight")
plt.show()


leaf_sizes = [len(leaf["indices"]) for leaf in trees[(4, 3)][0]["leaves"]]
plt.figure(figsize=(12, 3.5))
plt.bar(range(len(leaf_sizes)), leaf_sizes)
plt.xlabel("leaf number (visual word)")
plt.ylabel("number of descriptors")
plt.savefig("figures/descriptors_per_leaf.png", dpi=200, bbox_inches="tight")
plt.show()


top1 = [results_4a[t][0] * 100 for t in tree_list]
top5 = [results_4a[t][1] * 100 for t in tree_list]
plt.figure(figsize=(7, 4))
bars1 = plt.bar(x - 0.2, top1, 0.4, label="top-1")
bars5 = plt.bar(x + 0.2, top5, 0.4, label="top-5")
plt.bar_label(bars1, fmt="%.0f%%")
plt.bar_label(bars5, fmt="%.0f%%")
plt.xticks(x, [f"b={b}, depth={d}" for b, d in tree_list])
plt.ylabel("recall (%)")
plt.ylim(0, 105)
plt.legend()
plt.savefig("figures/recall_trees.png", dpi=200, bbox_inches="tight")
plt.show()


tree_cost = [b * d for b, d in tree_list]
flat_cost = [b ** d for b, d in tree_list]
plt.figure(figsize=(7, 4))
bars_f = plt.bar(x - 0.2, flat_cost, 0.4, label="flat vocabulary (b^depth)", color="0.6")
bars_t = plt.bar(x + 0.2, tree_cost, 0.4, label="vocabulary tree (b x depth)")
plt.bar_label(bars_f)
plt.bar_label(bars_t)
plt.yscale("log")
plt.ylim(1, 5e5)
plt.xticks(x, [f"b={b}, depth={d}" for b, d in tree_list])
plt.ylabel("distances per query descriptor (log)")
plt.legend(loc="upper left")
plt.savefig("figures/distance_calculations.png", dpi=200, bbox_inches="tight")
plt.show()